In [1]:
from pathlib import Path

from modules import SequenceRepresentation as sr
from modules import training

2025-03-13 15:19:08.434232: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-13 15:19:08.563846: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-13 15:19:09.120819: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-13 15:19:11.251465: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Matplotlib is building the font cache; this may take a moment.


In [15]:
#wd = Path("/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/test/")
wd = Path("/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
experiment_dirs = [f for f in wd.iterdir() if f.is_dir() and not (f.name.startswith("test") or f.name.startswith("slurm"))]
print(experiment_dirs[:max(3, len(experiment_dirs))])

[PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562Atf3UniPk.narrowPeak')]


In [16]:
def evaluate(experiment_dirs, evaluator_path, neg_evaluator_path = None):
    glob_seqs = {}
    glob_seqs_neg = {}
    for ed in experiment_dirs:
        if not (ed / evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / evaluator_path} does not exist")
            continue

        if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / neg_evaluator_path} does not exist")
            continue
        
        assert (ed / 'test_sequences_0.json').exists()
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
        # retrieve peaks from test data (peaks are stored as genomic elements in the sequences)
        peaks = {}
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in peaks:
                        peaks[peaksrc] = {}
                    if s.id not in peaks[peaksrc]:
                        peaks[peaksrc][s.id] = []
                    peakstart, peakend = e.getRelativePositions(s)
                    assert peakend == peakstart + 1, f"Peak {e} is not a single base pair"
                    peaks[peaksrc][s.id].append(peakstart)

        # store how many times and where each sequence was hit
        seqdict = {s.id: [] for g in testdata for s in g} 
        evaluator = training.loadMultiTrainingEvaluation(str(ed / evaluator_path), testdata)
        assert len(evaluator.trainings) == 1
        tr = evaluator.trainings[0]
        for link in tr.links:
            for occs in link.occs: # list of list of occurrences
                for occ in occs:
                    assert occ.sequence.id in seqdict
                    seqdict[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        if neg_evaluator_path is not None:
            # store how many times and where each sequence was hit
            assert (ed / 'negative_test_sequences_0.json').exists()
            testdata_neg = sr.loadJSONGenomeList(str(ed / 'negative_test_sequences_0.json'))
            seqdict_neg = {s.id: [] for g in testdata_neg for s in g} 
            evaluator_neg = training.loadMultiTrainingEvaluation(str(ed / neg_evaluator_path), testdata_neg)
            assert len(evaluator_neg.trainings) == 1
            tr = evaluator_neg.trainings[0]
            for link in tr.links:
                for occs in link.occs: # list of list of occurrences
                    for occ in occs:
                        assert occ.sequence.id in seqdict_neg
                        seqdict_neg[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        # globally count how many times each sequence was hit
        for sid in seqdict:
            if sid not in glob_seqs:
                glob_seqs[sid] = {'hits': 0, 'peaks': {}}
            glob_seqs[sid]['hits'] += len(seqdict[sid])
            for peaksrc in peaks:
                if sid in peaks[peaksrc]:
                    if peaksrc not in glob_seqs[sid]['peaks']:
                        glob_seqs[sid]['peaks'][peaksrc] = {'peaks': set(), 'hits': set()}
                    glob_seqs[sid]['peaks'][peaksrc]['peaks'].update(peaks[peaksrc][sid])
                    for p in peaks[peaksrc][sid]:
                        for hit in seqdict[sid]:
                            if hit[0] <= p < hit[1]:
                                glob_seqs[sid]['peaks'][peaksrc]['hits'].add(p)

        if neg_evaluator_path is not None:
            for sid in seqdict_neg:
                if sid not in glob_seqs_neg:
                    glob_seqs_neg[sid] = {'hits': 0}
                glob_seqs_neg[sid]['hits'] += len(seqdict_neg[sid])

    nseqs = len(glob_seqs.keys())
    nseqs_hit = len([k for k in glob_seqs.keys() if glob_seqs[k]['hits'] > 0])
    nmatches = sum([v['hits'] for v in glob_seqs.values()])

    print(f"Number of sequences: {nseqs}")
    print(f"Number of sequences with hits: {nseqs_hit} | ratio: {nseqs_hit/nseqs:.2f}")
    print(f"Number of matches: {nmatches} | ratio: {nmatches/nseqs:.2f}")
    print()

    for peaksrc in peaks:
        print(f"Peak source: {peaksrc}")
        n_peak_seqs = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks']])
        n_peak_seqs_with_hits = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks'] and glob_seqs[k]['peaks'][peaksrc]['hits']])
        n_peaks = sum([len(v) for v in peaks[peaksrc].values()])
        n_peaks_hit = sum([len(v['peaks'][peaksrc]['hits']) for v in glob_seqs.values() if peaksrc in v['peaks']])

        print(f"Number of sequences with peaks: {n_peak_seqs}")
        print(f"Number of sequences with hits on peaks: {n_peak_seqs_with_hits} | ratio: {n_peak_seqs_with_hits/n_peak_seqs:.2f}")
        print(f"Number of peaks: {n_peaks}")
        print(f"Number of hits on peaks: {n_peaks_hit} | ratio: {n_peaks_hit/n_peaks:.2f}")
        print()

    if neg_evaluator_path is not None:
        nseqs_neg = len(glob_seqs_neg.keys())
        nseqs_hit_neg = len([k for k in glob_seqs_neg.keys() if glob_seqs_neg[k]['hits'] > 0])
        nmatches_neg = sum([v['hits'] for v in glob_seqs_neg.values()])

        print(f"Number of negative sequences: {nseqs_neg}")
        print(f"Number of negative sequences with hits: {nseqs_hit_neg} | ratio: {nseqs_hit_neg/nseqs_neg:.2f}")
        print(f"Number of negative matches: {nmatches_neg} | ratio: {nmatches_neg/nseqs_neg:.2f}")
        print()

In [19]:

evaluate(experiment_dirs, 'evaluator_test.json', 'evaluator_negative_test.json')

Number of sequences: 370
Number of sequences with hits: 75 | ratio: 0.20
Number of matches: 178 | ratio: 0.48

Peak source: bed.tsv
Number of sequences with peaks: 370
Number of sequences with hits on peaks: 4 | ratio: 0.01
Number of peaks: 370
Number of hits on peaks: 4 | ratio: 0.01

Peak source: fimo.tsv
Number of sequences with peaks: 31
Number of sequences with hits on peaks: 0 | ratio: 0.00
Number of peaks: 31
Number of hits on peaks: 0 | ratio: 0.00

Peak source: mast.tsv
Number of sequences with peaks: 27
Number of sequences with hits on peaks: 0 | ratio: 0.00
Number of peaks: 28
Number of hits on peaks: 0 | ratio: 0.00

Number of negative sequences: 370
Number of negative sequences with hits: 23 | ratio: 0.06
Number of negative matches: 48 | ratio: 0.13



In [12]:
evaluate(experiment_dirs, 'STREME/streme_evaluator_dummymodel_test.json')

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/wgEncodeAwgTfbsSydhK562Bhlhe40nb100IggrabUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/wgEncodeAwgTfbsSydhK562Bhlhe40nb100IggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
Number of sequences: 9632
Number of sequences with hits: 1393 | ratio: 0.14
Number of matches: 2978 | ratio: 0.31

Peak source: bed.tsv
Number of sequences with peaks: 9632
Number of sequences with hits on peaks: 356 | ratio: 0.04
Number of peaks: 4460
Number of hits on peaks: 356 | ratio: 0.08

Peak source: fimo.tsv
Number of sequences with peaks: 1504
Number of sequences with hits on peaks: 21 | ratio: 0.01
Number of peaks: 784
Number of hits on peaks: 21 | ratio: 0.03

Peak source: mast.tsv
Number of sequences with peaks: 1502
Number of sequences with hits on peaks: 24 | ratio: 0.02
Number of peaks: 874
Number of hits on peaks: 24 | ratio: 0.03

